In [2]:
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. 模拟一些私有数据（实际上这里可以写代码读取PDF或Excel）
# 这是一个列表，每一项是一段话。在实际项目中，你需要把长文章切分成这样的小段（Chunking）。
text_data = [
    "贝叶斯统计学是一种基于贝叶斯定理的统计推断方法。",
    "先验概率（Prior Probability）是在考虑新证据之前，对事件发生可能性的初始评估。",
    "后验概率（Posterior Probability）是在考虑新证据之后，更新后的事件发生可能性。",
    "最大似然估计（MLE）是频率学派的方法，与贝叶斯学派不同，它不考虑先验分布。",
    "大模型应用开发中，RAG技术可以有效解决模型幻觉问题。"
]

# 把文本转换成 LangChain 的 Document 对象
documents = [Document(page_content=t) for t in text_data]

print("数据准备完毕，共准备了", len(documents), "条知识。")

# 2. 初始化向量模型 (Embedding Model)
# 这个模型负责把文字变成数字向量。我们用一个免费、小巧的本地模型。
# 第一次运行会自动下载模型（约400MB），请耐心等待。
print("正在加载向量化模型（第一次运行可能需要下载）...")
# 使用智源的中文小模型 BGE-Small-ZH
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")

# 3. 构建向量数据库 (Vector Database)
# 这一步相当于把所有文字变成了坐标系里的点，并存起来
vector_store = FAISS.from_documents(documents, embeddings)

print("知识库构建完成！已将文本转化为向量存储。")

数据准备完毕，共准备了 5 条知识。
正在加载向量化模型（第一次运行可能需要下载）...


C:\Users\38244\AppData\Local\Temp\ipykernel_42828\116275491.py:26: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


知识库构建完成！已将文本转化为向量存储。


In [3]:
# 假设用户问了一个问题
query = "什么是先验概率？"

# 在数据库里找和这个问题最相似的2句话 (k=2)
# 类似于 KNN 算法寻找最近邻
docs = vector_store.similarity_search(query, k=2)

print(f"\n用户提问: {query}")
print("-" * 30)
print("检索到的相关知识:")
for i, doc in enumerate(docs):
    print(f"{i+1}. {doc.page_content}")


用户提问: 什么是先验概率？
------------------------------
检索到的相关知识:
1. 先验概率（Prior Probability）是在考虑新证据之前，对事件发生可能性的初始评估。
2. 后验概率（Posterior Probability）是在考虑新证据之后，更新后的事件发生可能性。


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# === 这里填入你申请的 API Key ===
# 如果你是 DeepSeek:
API_KEY = "sk-43401194122e4d4f8645889b070af0df"
BASE_URL = "https://api.deepseek.com" 

# 初始化大模型
llm = ChatOpenAI(
    model="deepseek-chat", # 或者 "deepseek-coder"
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.1 # 温度越低，回答越严谨
)

# 定义一个 Prompt (提示词模板)
# 告诉模型：你是一个助手，只能根据我给你的 context 回答 question
prompt_template = ChatPromptTemplate.from_template("""
你是一个统计学助教。请基于下面的【已知信息】回答用户的【问题】。
如果已知信息里没有答案，就说不知道，不要编造。

【已知信息】:
{context}

【问题】:
{question}
""")

# 定义整个处理流程 (Chain)
# 流程：检索 -> 组装Prompt -> 发给LLM -> 解析结果
def ask_question(question):
    # 1. 检索
    related_docs = vector_store.similarity_search(question, k=2)
    # 把检索到的文本拼成一个字符串
    context_text = "\n\n".join([d.page_content for d in related_docs])
    
    # 2. 组装 Chain
    chain = prompt_template | llm | StrOutputParser()
    
    # 3. 运行
    response = chain.invoke({"context": context_text, "question": question})
    return response

# --- 测试 ---
my_question = "先验概率和后验概率有什么区别？"
print(f"\n正在思考问题: {my_question} ...")
answer = ask_question(my_question)
print("\n=== AI 回答 ===")
print(answer)


正在思考问题: 先验概率和后验概率有什么区别？ ...

=== AI 回答 ===
根据已知信息，两者的区别在于：

1. **先验概率**：是在考虑任何新证据或信息之前，对事件发生可能性的初始评估或估计。
2. **后验概率**：是在结合了新的证据或信息之后，对同一事件发生可能性更新后的评估。

简单来说，先验概率是“事前”的初始判断，后验概率是结合新证据后“事后”的更新判断。
